# Final TrackMate Robust Z viewer

`TrackMate_final_r2p5_q150` の全40シリーズ・Slice 1–180を確認します。

- LoG radius 2.5、Median OFF、Q ≥ 150
- Contrast/SNRの負値を0として全特徴量をlog1p変換
- hemisphere（MAY17Rなど）内でd/vを統合してMedian/MAD Robust Z化
- 共通Robust Z閾値の積集合、ランダム標本、Zoom・XY移動に対応


In [6]:
from functools import lru_cache
from pathlib import Path
import xml.etree.ElementTree as ET
import zlib

import ipywidgets as widgets
from IPython.display import clear_output, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile as tiff

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULT_ROOT = PROJECT_ROOT / 'outputs' / 'TrackMate_final_r2p5_q150'
SUMMARY_PATH = RESULT_ROOT / 'robust_z_slice_summary.csv'
if SUMMARY_PATH.is_file():
    SUMMARY = pd.read_csv(str(SUMMARY_PATH))
    SUMMARY_SOURCE = 'complete summary'
else:
    partial_paths = sorted((RESULT_ROOT / 'partial_summaries').glob('*_slices.csv'))
    if not partial_paths:
        raise FileNotFoundError('Robust Z output is not available yet: {}'.format(SUMMARY_PATH))
    SUMMARY = pd.concat([pd.read_csv(str(path)) for path in partial_paths], ignore_index=True)
    SUMMARY_SOURCE = 'partial summaries ({}/20 blocks complete)'.format(len(partial_paths))
FEATURES = [
    ('Contrast', 'ROBUST_Z_CONTRAST_CH1', (0.0, 10.0), True),
    ('SNR', 'ROBUST_Z_SNR_CH1', (0.0, 10.0), True),
    ('SD', 'ROBUST_Z_STD_INTENSITY_CH1', (-2.0, 6.0), True),
    ('CV', 'ROBUST_Z_CV', (-2.0, 5.0), True),
    ('Mean', 'ROBUST_Z_MEAN_INTENSITY_CH1', (-3.0, 6.0), True),
    ('Min', 'ROBUST_Z_MIN_INTENSITY_CH1', (-3.0, 3.5), True),
    ('Quality', 'ROBUST_Z_QUALITY', (-3.0, 10.0), False),
    ('Max', 'ROBUST_Z_MAX_INTENSITY_CH1', (-3.0, 10.0), False),
    ('Median', 'ROBUST_Z_MEDIAN_INTENSITY_CH1', (-3.0, 10.0), False),
    ('Total', 'ROBUST_Z_TOTAL_INTENSITY_CH1', (-3.0, 10.0), False),
]
DATASETS = sorted(SUMMARY['dataset'].unique())
SLICES = list(range(1, 181))
INDEX = {(row['dataset'], int(row['slice'])): row for _, row in SUMMARY.iterrows()}
if len(SUMMARY) != len(DATASETS) * 180 or len(INDEX) != len(SUMMARY):
    raise RuntimeError('Dataset/slice grid is incomplete.')

def current_path(path_string):
    path = Path(path_string)
    parts = list(path.parts)
    if 'outputs' in parts:
        return PROJECT_ROOT.joinpath(*parts[parts.index('outputs'):])
    return path

def row_for(dataset, slice_number):
    return INDEX[(dataset, int(slice_number))]

print('Datasets:', len(DATASETS), '| Images:', len(INDEX), '| Source:', SUMMARY_SOURCE)


Datasets: 40 | Images: 7200 | Source: complete summary


In [7]:
@lru_cache(maxsize=6)
def read_spots(xml_path_string, expected_count):
    spots = np.empty((int(expected_count), 2 + len(FEATURES)), dtype=np.float32)
    index = 0
    for _, element in ET.iterparse(xml_path_string, events=('end',)):
        if element.tag == 'Spot':
            spots[index, 0] = float(element.attrib['POSITION_X'])
            spots[index, 1] = float(element.attrib['POSITION_Y'])
            for feature_index, (_, attribute, _, _) in enumerate(FEATURES):
                spots[index, 2 + feature_index] = float(element.attrib[attribute])
            index += 1
        element.clear()
    if index != int(expected_count):
        raise RuntimeError('Spot count mismatch: expected {}, read {}'.format(expected_count, index))
    return spots

@lru_cache(maxsize=4)
def read_image(path_string):
    return tiff.imread(path_string)

def sample_spots(spots, active_filters, sample_n, seed):
    mask = np.ones(len(spots), dtype=bool)
    for feature_index, lower, upper, _ in active_filters:
        values = spots[:, 2 + feature_index]
        mask &= (values > lower) & (values < upper)
    eligible = np.flatnonzero(mask)
    eligible_count = len(eligible)
    if eligible_count > int(sample_n):
        rng = np.random.RandomState(int(seed) & 0xffffffff)
        eligible = rng.choice(eligible, size=int(sample_n), replace=False)
    return spots[eligible], eligible_count

def view_bounds(center, full_size, zoom):
    size = float(full_size) / float(zoom)
    lower = min(max(float(center) - size / 2.0, 0.0), max(float(full_size) - size, 0.0))
    return lower, lower + size


In [ ]:
dataset_widget = widgets.Dropdown(options=DATASETS, value=DATASETS[0], description='Dataset:', layout=widgets.Layout(width='55%'))
slice_widget = widgets.IntSlider(value=80, min=1, max=180, step=1, description='Slice:', continuous_update=False, layout=widgets.Layout(width='70%'))
previous_slice = widgets.Button(description='← Slice')
next_slice = widgets.Button(description='Slice →')
previous_dataset = widgets.Button(description='← Dataset')
next_dataset = widgets.Button(description='Dataset →')
sample_widget = widgets.BoundedIntText(value=10000, min=100, max=100000, step=100, description='Random N:')
marker_widget = widgets.FloatSlider(value=5, min=1, max=30, step=1, description='Marker:', continuous_update=False)
alpha_widget = widgets.FloatSlider(value=0.7, min=0.1, max=1, step=0.1, description='Alpha:', continuous_update=False)
color_widget = widgets.Dropdown(options=['lime', 'red', 'cyan', 'yellow', 'magenta'], value='lime', description='Color:')
contrast_widget = widgets.FloatRangeSlider(value=(0.5, 99.7), min=0, max=100, step=0.1, description='Percentile:', continuous_update=False, layout=widgets.Layout(width='75%'))
zoom_widget = widgets.SelectionSlider(options=[0.5, 0.75, 1, 1.5, 2, 3, 4, 6, 8, 12, 16], value=1, description='Zoom:', continuous_update=False)
center_x_widget = widgets.IntSlider(value=2048, min=0, max=4095, description='Center X:', continuous_update=False, layout=widgets.Layout(width='80%'))
center_y_widget = widgets.IntSlider(value=1080, min=0, max=2159, description='Center Y:', continuous_update=False, layout=widgets.Layout(width='80%'))
figure_widget = widgets.FloatSlider(value=11, min=5, max=16, step=0.5, description='Figure:', continuous_update=False)
reset_button = widgets.Button(description='Reset view')
refresh_button = widgets.Button(description='Refresh')
status_widget = widgets.HTML()
viewer_output = widgets.Output()

filter_widgets = []
filter_rows = []
for index, (label, _, initial_bounds, initial_enabled) in enumerate(FEATURES):
    enabled = widgets.Checkbox(value=initial_enabled, description=label, indent=False, layout=widgets.Layout(width='115px'))
    bounds = widgets.FloatRangeSlider(value=initial_bounds, min=-10, max=10, step=0.1, description='Z:', disabled=not initial_enabled, continuous_update=False, readout_format='.1f', layout=widgets.Layout(width='75%'))
    filter_widgets.append((index, label, enabled, bounds))
    filter_rows.append(widgets.HBox([enabled, bounds]))
filter_accordion = widgets.Accordion(children=[widgets.VBox(filter_rows)])
filter_accordion.set_title(0, 'Common Robust Z filters')

def active_filters():
    result = []
    for index, label, enabled, bounds in filter_widgets:
        if enabled.value:
            lower, upper = bounds.value
            result.append((index, float(lower), float(upper), label))
    return result

def reset_view(_=None, render=True):
    row = row_for(dataset_widget.value, slice_widget.value)
    image = read_image(str(current_path(row['input_tif']).resolve()))
    height, width = image.shape
    center_x_widget.max = width - 1
    center_y_widget.max = height - 1
    center_x_widget.value = width // 2
    center_y_widget.value = height // 2
    zoom_widget.value = 1
    if render:
        render_view()

def render_view(_=None):
    try:
        dataset = dataset_widget.value
        slice_number = int(slice_widget.value)
        row = row_for(dataset, slice_number)
        image = read_image(str(current_path(row['input_tif']).resolve()))
        spots = read_spots(str(current_path(row['robust_z_xml']).resolve()), int(row['spots_q150']))
        filters = active_filters()
        signature = tuple((i, lo, hi) for i, lo, hi, _ in filters)
        seed = zlib.crc32('{}|{}|{}'.format(dataset, slice_number, signature).encode('utf-8')) & 0xffffffff
        shown, eligible_count = sample_spots(spots, filters, sample_widget.value, seed)
        vmin, vmax = np.percentile(image, contrast_widget.value)
        if vmax <= vmin:
            vmax = vmin + 1
        height, width = image.shape
        zoom = float(zoom_widget.value)
        x0, x1 = view_bounds(center_x_widget.value, width, zoom)
        y0, y1 = view_bounds(center_y_widget.value, height, zoom)
        fw = float(figure_widget.value)
        fh = min(11, max(3, fw * (y1 - y0) / max(x1 - x0, 1)))
        fig, ax = plt.subplots(figsize=(fw, fh), dpi=120)
        ax.imshow(image, cmap='gray', vmin=vmin, vmax=vmax, origin='upper')
        ax.scatter(shown[:, 0], shown[:, 1], s=marker_widget.value, facecolors='none', edgecolors=color_widget.value, linewidths=0.7, alpha=alpha_widget.value)
        ax.set_xlim(x0, x1)
        ax.set_ylim(y1, y0)
        ax.set_title('{} | Slice {:03d} | Q≥150 | zoom {}x'.format(dataset, slice_number, zoom_widget.value))
        ax.set_xlabel('X (pixel)')
        ax.set_ylabel('Y (pixel)')
        fig.tight_layout()
        selected_saved = int(row['spots_selected'])
        status_widget.value = '<b>{}</b> Slice {:03d} | Q≥150 {:,} | current eligible {:,} | shown {:,} | initially saved {:,}'.format(dataset, slice_number, len(spots), eligible_count, len(shown), selected_saved)
        with viewer_output:
            clear_output(wait=True)
            display(fig)
            plt.close(fig)
    except Exception as error:
        status_widget.value = '<b style="color:red">{}: {}</b>'.format(type(error).__name__, error)

def change_filter(change, bounds):
    bounds.disabled = not bool(change['new'])
    render_view()

def shift_dataset(delta):
    index = (DATASETS.index(dataset_widget.value) + delta) % len(DATASETS)
    dataset_widget.value = DATASETS[index]

previous_slice.on_click(lambda _: setattr(slice_widget, 'value', max(1, slice_widget.value - 1)))
next_slice.on_click(lambda _: setattr(slice_widget, 'value', min(180, slice_widget.value + 1)))
previous_dataset.on_click(lambda _: shift_dataset(-1))
next_dataset.on_click(lambda _: shift_dataset(1))
reset_button.on_click(reset_view)
refresh_button.on_click(render_view)
dataset_widget.observe(lambda _: reset_view(), names='value')
slice_widget.observe(render_view, names='value')
for widget in [sample_widget, marker_widget, alpha_widget, color_widget, contrast_widget, zoom_widget, center_x_widget, center_y_widget, figure_widget]:
    widget.observe(render_view, names='value')
for _, _, enabled, bounds in filter_widgets:
    enabled.observe(lambda change, bounds=bounds: change_filter(change, bounds), names='value')
    bounds.observe(render_view, names='value')

controls = widgets.VBox([
    widgets.HBox([dataset_widget, previous_dataset, next_dataset]),
    widgets.HBox([slice_widget, previous_slice, next_slice]),
    widgets.HBox([sample_widget, refresh_button]),
    filter_accordion,
    widgets.HBox([marker_widget, alpha_widget, color_widget]),
    contrast_widget,
    widgets.HBox([zoom_widget, figure_widget, reset_button]),
    center_x_widget,
    center_y_widget,
    status_widget,
])
reset_view(render=False)
display(controls, viewer_output)
render_view()


Output()